<div style="border: 3px solid #b42318; background: #fef3f2; color: #7a271a; border-radius: 12px; padding: 16px 20px; line-height: 1.5">
<div style="font-size: 20px; font-weight: 800; color: #b42318; margin-bottom: 10px">
&#9888;&#65039; ЧЕРНОВИК &mdash; НЕ АКТУАЛЬНАЯ ВЕРСИЯ
</div>
<p style="margin: 0 0 10px 0"><b>Это занятие ещё в работе и будет переписано.</b>
Формулировки, данные и порядок заданий изменятся; часть материала может
опираться на то, что к этому моменту курса ещё не прочитано.</p>
<p style="margin: 0">Заниматься по нему пока не нужно &mdash; дождитесь
окончательной версии. Актуально сейчас только <b>занятие&nbsp;1</b>.</p>
</div>

# Домашняя работа 7. Обобщённый метрический классификатор и STOLP

**Курс «Машинное обучение», 4 курс**

| | |
|---|---|
| К лабораторной | занятие 7 — Метрические и байесовские методы классификации |
| Опора | материал семинара 7 и лекций до него |
| Ожидаемое время | 3–4 часа |
| Данные | **ваша индивидуальная таблица** (по ФИО) |

На занятии мы пользовались `KNeighborsClassifier`. Дома выяснится, что kNN, парзеновские окна и метод потенциальных функций — это **одна формула** с разной весовой функцией: реализовав её один раз, вы получите все пять методов сразу. Во второй задаче — отбор эталонов, который сокращает выборку в десятки раз, не теряя качества.

> **Чем это отличается от занятия.** На семинаре данные были учебные и общие —
> так удобно разбирать. Дома данные ваши: таблица порождается по ФИО, и ни у
> кого в группе она не повторяется. Приёмы те же, числа другие — поэтому
> отвечать придётся за свои числа, а не за преподавательские.


## Как устроена работа

Работа делится на две части, и делятся они по назначению, а не по сложности.

**Обязательная часть — допуск.** Без неё работа не принимается: это тот минимум,
без которого занятие считается неусвоенным. Здесь всегда есть хотя бы одна
реализация «с нуля», сверенная с `scikit-learn` численно.

**Часть на оценку** (помечена значком ★). Она не нужна для допуска — но балл
за работу выставляется именно по ней, и каждый выполненный пункт идёт в зачёт
отдельно. Браться стоит даже за один пункт: это лучше, чем не браться вовсе.


## Что нужно сдать

Заполненный ноутбук, в котором:

1. выполнены все ячейки с `# TODO` в обязательной части, код исполняется сверху
   вниз без ошибок в свежем ядре (Kernel → Restart & Run All);
2. под каждым заданием заполнена ячейка **Вывод** — своими словами,
   со ссылкой на полученные числа;
3. графики подписаны: заголовок, оси, легенда;
4. в ячейке варианта вписано ваше ФИО.

> Списывание видно сразу: у каждого студента свой датасет и свой набор методов.

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/ml_labs/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant, submission_name  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from labdata import load_personal
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

## Индивидуальный вариант

Впишите своё ФИО (или почту) в переменную `STUDENT` — вариант вычисляется детерминированно, при повторном запуске он тот же самый.

Регистр, лишние пробелы и написание «ё»/«е» роли не играют. Если ФИО вписано неверно, вариант будет чужим — проверьте вывод ячейки.

In [ ]:
STUDENT = "Фамилия Имя Отчество"   # <-- впишите себя

variant = get_variant(STUDENT, lab=7)
describe_variant(variant)

print("\nСдавать под именем:", submission_name(STUDENT, lab=7))

---
# Задача 1. Пять методов — одна формула

Определение 6.1 задаёт классификатор через веса соседей:

$$
a(x) = \arg\max_k\sum_{i}\bigl[y_{(i)} = k\bigr]\,w(i, x).
$$

| метод | вес $w(i,x)$ |
|---|---|
| $k$ ближайших соседей | $[i \le k]$ |
| kNN с весами | $[i \le k]\,q^{\,i}$, $q\in(0,1)$ |
| парзеновское окно ширины $h$ | $K\bigl(\rho(x,x_{(i)})/h\bigr)$ |
| парзеновское окно переменной ширины | $K\bigl(\rho(x,x_{(i)})/\rho(x,x_{(k+1)})\bigr)$ |
| потенциальные функции | $\gamma_i\,K\bigl(\rho(x,x_i)/h\bigr)$ |

Ваш вариант: `variant["metric_method"]`, ядро `variant["kernel"]`,
метрика `variant["distance"]`. Реализуйте **один** класс с параметром-методом.

> **Напоминание — NumPy: `argsort` и ранг соседа.** `np.argsort(a)` возвращает не отсортированный массив, а **номера** элементов в
> порядке возрастания: `argsort([30, 10, 20])` даёт `[1, 2, 0]` — «сначала
> элемент 1, потом 2, потом 0». С `axis=1` то же делается построчно.
>
> Дальше используется приём, который стоит запомнить: `argsort` от `argsort` даёт
> **ранг** каждого элемента, то есть его место в очереди. Если `D[j, i]` —
> расстояние от объекта $j$ до объекта $i$, то `rank[j, i] = 0` означает «$i$ —
> ближайший сосед для $j$», `rank[j, i] = 1` — второй по близости и так далее.
> Условие «взять $k$ ближайших» превращается в одну строку `rank < k`, без
> сортировок в цикле.

In [ ]:
KERNELS = {
    "прямоугольное": lambda r: (np.abs(r) <= 1) * 0.5,
    "треугольное": lambda r: np.maximum(0.0, 1 - np.abs(r)),
    "Епанечникова": lambda r: np.maximum(0.0, 0.75 * (1 - r ** 2)),
    "квартическое": lambda r: np.maximum(0.0, (15 / 16) * (1 - r ** 2) ** 2),
    "гауссовское": lambda r: np.exp(-0.5 * r ** 2) / np.sqrt(2 * np.pi),
}

# TODO: словарь DISTANCES с четырьмя метриками (евклидова -- через тождество
#       из занятия 1, чтобы не было тройного цикла).

class MetricClassifier:
    """Обобщённый метрический классификатор (опр. 6.1).

    Один класс, пять методов -- различаются ТОЛЬКО функцией веса w(i, x):
      'knn'             : [i <= k]
      'knn_weighted'    : [i <= k] q^i
      'parzen_fixed'    : K(rho / h)
      'parzen_variable' : K(rho / rho_(k+1))
      'potential'       : gamma_i K(rho / h)
    Подсказка: номер соседа i получается двойным argsort матрицы расстояний.
    Класс должен наследоваться от sklearn.base.BaseEstimator и ClassifierMixin,
    иначе его нельзя передать в cross_val_score.
    Нужны методы: fit(X, y, gamma=None), decision(X), predict(X).
    """
    # TODO


METHOD_KEY = {
    "kNN с равными весами": "knn",
    "kNN с весами w(i) = q^i": "knn_weighted",
    "парзеновское окно фиксированной ширины": "parzen_fixed",
    "парзеновское окно переменной ширины": "parzen_variable",
    "метод потенциальных функций": "potential",
}
my_method = METHOD_KEY[variant["metric_method"]]

### Задание 1.1. Все пять на одной картинке

Сверьте свой `knn` со `sklearn` (предсказания обязаны совпасть полностью)
и покажите границы решения всех пяти методов при одном и том же ядре.

In [ ]:
Xm, ym = make_blobs(n_samples=220, centers=3, cluster_std=1.6, random_state=RANDOM_STATE)
Xm = StandardScaler().fit_transform(Xm)
Xa, Xv, ya, yv = train_test_split(Xm, ym, test_size=0.35, stratify=ym,
                                  random_state=RANDOM_STATE)

# TODO: 1) сверьте свой knn(5) с KNeighborsClassifier(5) -- предсказания
#          должны совпасть полностью;
#       2) постройте границы решения всех пяти методов на одном рисунке
#          (knn k=5; knn_weighted k=15, q=0.75; parzen_fixed h=0.6;
#           parzen_variable k=10; potential h=0.6).

> **Вывод.** Чем отличаются границы разных весовых функций? Что происходит с парзеновским окном фиксированной ширины в разреженной области выборки?
>
> *(ваш ответ здесь)*

### Задание 1.2. Ваша реализация на вашей выборке

Двумерная синтетика удобна тем, что видно границу. Но работать классификатор
обязан на табличных данных, где смотреть не на что. Загрузите свою выборку и
убедитесь, что ваш `knn` совпадает со `sklearn` **на всех объектах** — это
проверка корректности, а не «примерно похоже».

In [ ]:
# TODO: 1) загрузите свою выборку через load_personal(variant); если вариант
#          регрессионный -- бинаризуйте цель по медиане обучающей части;
#       2) обучите свой MetricClassifier(method="knn", k=5) и
#          KNeighborsClassifier(5);
#       3) напечатайте долю совпадающих предсказаний (обязана быть 1.0)
#          и accuracy обеих реализаций.

> **Вывод.** Совпали ли предсказания полностью? Если нет — где искать ошибку?
>
> *(ваш ответ здесь)*

---

> ### ★ Дальше — часть на оценку
>
> Обязательная часть закончилась: если вы дошли досюда и всё работает, работа
> будет принята. Дальше идут задания, по которым выставляется балл. Каждый
> пункт засчитывается отдельно, поэтому имеет смысл сделать хотя бы один.

# Задача 2★. Отступ и отбор эталонов (STOLP)

Определение 6.4: отступ объекта в метрическом классификаторе —

$$
M(x_i) = \Gamma_{y_i}(x_i) - \max_{k \ne y_i}\Gamma_k(x_i).
$$

По величине отступа объекты делятся на эталонные ($M \gg 0$), неинформативные,
пограничные ($M \approx 0$), ошибочные ($M < 0$) и шумовые ($M \ll 0$).
Алгоритм STOLP отбирает небольшое подмножество эталонов, выбрасывая шум
и «неинформативную массовку».

In [ ]:
def margins(clf, X, y):
    """M(x_i) = Gamma_{y_i}(x_i) - max_{k != y_i} Gamma_k(x_i).

    Подсказка: возьмите clf.decision(X), выньте столбец своего класса,
    в копии матрицы поставьте -inf на месте своего класса и возьмите максимум.
    """
    raise NotImplementedError


def stolp(X, y, delta=0.0, max_errors=0.01, k=5, kernel="Епанечникова"):
    """STOLP:
       1) посчитать отступы по классификатору на ПОЛНОЙ выборке и выбросить
          объекты с M < delta (это шум);
       2) начать множество эталонов с одного объекта на класс -- с максимальным
          отступом;
       3) пока доля ошибок правила ближайшего эталона больше max_errors,
          добавлять ошибочно классифицированный объект (берите тот, у которого
          отступ по полной выборке наибольший -- он представляет целую область).
       Возвращает (индексы эталонов, отступы).
    """
    raise NotImplementedError

### Задание 2.1. Сколько объектов на самом деле нужно

Возьмите выборку, испортите метки у части объектов (так бывает в реальной
разметке) и посмотрите, что отберёт STOLP.

Важно: обучающая и контрольная части должны быть из одного распределения —
порождайте одну выборку и делите её, а не вызывайте генератор дважды
с разными `random_state`.

In [ ]:
X_all, y_all = make_blobs(n_samples=900, centers=3, cluster_std=2.2,
                          random_state=RANDOM_STATE)
X_all = StandardScaler().fit_transform(X_all)
X_st, Xq, y_st, yq = train_test_split(X_all, y_all, train_size=300, stratify=y_all,
                                      random_state=RANDOM_STATE)
noise_idx = np.random.default_rng(0).choice(len(y_st), 18, replace=False)
y_st = y_st.copy()
y_st[noise_idx] = (y_st[noise_idx] + 1) % 3

# TODO: 1) примените STOLP, выведите число эталонов и сколько среди них
#          объектов с испорченными метками;
#       2) сравните точность на ОТЛОЖЕННЫХ данных (Xq, yq): вся выборка
#          с kNN(5) против правила ближайшего эталона.

In [ ]:
# TODO: постройте профиль отступов (столбики, упорядоченные по возрастанию M,
#       отрицательные другим цветом) и картинку выборки с отмеченными
#       эталонами и испорченными метками.

> **Вывод.** Сколько объектов оставил STOLP и как изменилась точность? Попали ли испорченные метки в эталоны? Зачем вообще отбирать эталоны?
>
> *(ваш ответ здесь)*

### Задание 2.2★. STOLP на вашей выборке

Сожмите свою обучающую выборку алгоритмом STOLP (данные уже загружены в
задании 1.2) и посмотрите, сколько объектов осталось и что стало с качеством.

In [ ]:
# TODO: 1) примените stolp(...) к своей обучающей выборке (Xp_tr, yp_tr);
#       2) обучите kNN ТОЛЬКО на эталонах и сведите в таблицу три строки:
#          своя реализация на всей выборке, sklearn на всей выборке, своя
#          на эталонах -- вместе с числом объектов, которые модель хранит;
#       3) напечатайте степень сжатия и число объектов с отрицательным отступом.

> **Вывод.** Насколько сжалась выборка и что стало с качеством? Почему сжатая модель может оказаться **лучше** полной?
>
> *(ваш ответ здесь)*

## Итоги домашней работы

Кратко ответьте на вопросы:

1. Все пять методов — частные случаи одной формулы. У какого из них нет настраиваемых параметров вовсе и чем это удобно?
2. Вы применили STOLP и получили 3 эталона из 500 при точности 0.95. Насколько такому результату можно доверять и что стоит проверить?

## Обратная связь

Это не оценивается и на балл не влияет — нужно, чтобы поправить работу к
следующему году. Отвечайте одной строкой, честно.

| | |
|---|---|
| Сколько часов заняло | |
| Сложность от 1 до 5 | |
| Что осталось непонятным | |
| Какое задание показалось лишним | |

---

Проверьте перед сдачей: Kernel → Restart & Run All проходит без ошибок,
все ячейки **Вывод** заполнены, графики подписаны.

Имя файла — то, что напечатала ячейка с вариантом: `hw01_Ivanov_I_I.ipynb`.
Номер работы впереди, фамилия и инициалы латиницей. Если сдаёте исправленную
версию, допишите `_v2`.